# Momentum Transformer — Live Inference
Downloads fresh data up to **today**, runs the trained model, and shows:
- Positions for the next session, from the latest close, with actual portfolio weights
- The live track record since 2024 and the last 90 days vs the benchmark

> **Prerequisite:** upload your trained `best_model.pth` (e.g. as a Kaggle Dataset). Generated by `create_live_notebook.py`.

## 1. Install Dependencies

In [ ]:
!pip install -q yfinance pandas numpy torch tqdm scipy matplotlib

## 2. Load Trained Model

In [ ]:
import os, shutil, glob

os.makedirs('models', exist_ok=True)
if not os.path.exists('models/best_model.pth'):
    matches = glob.glob('/kaggle/input/**/*.pth', recursive=True) + glob.glob('/content/**/*.pth', recursive=True)
    if matches:
        shutil.copy(matches[0], 'models/best_model.pth')
        print(f'Model copied from: {matches[0]}')
    else:
        print('WARNING: best_model.pth not found. Upload it first.')
else:
    print('Using models/best_model.pth')


## 3. Source Code

In [ ]:
# ==================================================
# data.py
# ==================================================

import yfinance as yf
import pandas as pd
import os

# 20 Macro/Sector ETFs + 50 Mega-Cap Stocks = 70 highly liquid assets.
# Note: the stock list is today's mega caps, so historical backtests on it carry
# survivorship bias (see README).
TICKERS = [
    # Macro / sector ETFs
    'SPY', 'QQQ', 'IWM', 'EFA', 'EEM',
    'TLT', 'IEF', 'SHY', 'LQD', 'HYG',
    'GLD', 'SLV', 'USO', 'UNG', 'DBA',
    'VNQ', 'XLF', 'XLK', 'XLV', 'XLE',
    # Mega-cap stocks
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
    'JPM', 'V', 'PG', 'MA', 'HD', 'CVX', 'MRK', 'ABBV', 'LLY', 'PEP',
    'KO', 'BAC', 'AVGO', 'TMO', 'COST', 'MCD', 'CSCO', 'WMT', 'CRM', 'DIS',
    'PFE', 'DHR', 'NKE', 'XOM', 'TXN', 'UNP', 'LIN', 'ADBE', 'VZ', 'PM',
    'NEE', 'RTX', 'QCOM', 'INTC', 'AMD', 'HON', 'UPS', 'AMGN', 'IBM', 'BA'
]


def download_data(tickers, start_date, end_date, output_path=None):
    """
    Downloads OHLCV data for a list of tickers and returns it in long format
    (one row per Date x Ticker).
    """
    print(f"Downloading data for {len(tickers)} tickers from {start_date} to {end_date}...")
    # auto_adjust=True: Close is adjusted for splits and dividends. Older yfinance
    # versions defaulted to False (raw Close + separate 'Adj Close'), which would
    # turn a 4-for-1 split into a fake -75% one-day return. Pass it explicitly.
    data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)

    # yfinance returns MultiIndex columns (field, ticker) for several tickers;
    # stack the ticker level into rows to get long format.
    if len(tickers) > 1:
        data = data.stack(level=1).rename_axis(['Date', 'Ticker']).reset_index()
    else:
        data = data.reset_index()
        data['Ticker'] = tickers[0]

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        data.to_csv(output_path, index=False)
        print(f"Data saved to {output_path}")

    return data

# ==================================================
# features.py
# ==================================================

import pandas as pd
import numpy as np
import os

def compute_macd(close, short_window, long_window):
    """Compute MACD signal for a given short and long window."""
    ewm_short = close.ewm(span=short_window, adjust=False).mean()
    ewm_long = close.ewm(span=long_window, adjust=False).mean()
    return ewm_short - ewm_long

FEATURE_COLS = [
    'Return_1d_scaled', 'Return_21d_scaled', 'Return_63d_scaled',
    'Return_126d_scaled', 'Return_252d_scaled',
    'MACD_8_24_scaled', 'MACD_16_28_scaled', 'MACD_32_96_scaled'
]

def compute_features(df, keep_unlabeled=False):
    """
    Computes momentum features for each ticker.
    df is expected to have ['Date', 'Ticker', 'Close', 'Open', 'High', 'Low', 'Volume']

    keep_unlabeled=False (training/backtest): drop every row with a NaN, including
        each ticker's last row, whose next-day return is not known yet.
    keep_unlabeled=True (live trading): keep rows whose features are complete even
        if Target_Return is NaN, so the most recent day can still be traded.
    """
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
    
    processed_dfs = []
    
    for ticker, group in df.groupby('Ticker'):
        group = group.copy()
        
        # Daily return
        group['Return_1d'] = group['Close'].pct_change()
        
        # Ex-ante volatility: EWMA of squared returns, then sqrt (Paper Section VI.A)
        # sigma^2_t = lambda * sigma^2_{t-1} + (1-lambda) * r^2_{t-1}, lambda = 1 - 1/60
        lam = 1.0 - 1.0 / 60.0
        sq_rets = group['Return_1d'] ** 2
        ewma_var = sq_rets.ewm(alpha=1 - lam, adjust=False).mean()
        group['Vol_60d'] = np.sqrt(ewma_var)
        
        # Forward fill and drop leading NaNs for Vol
        group['Vol_60d'] = group['Vol_60d'].ffill()
        
        # Multi-horizon returns (1, 21, 63, 126, 252 days)
        horizons = [1, 21, 63, 126, 252]
        for h in horizons:
            # We use log returns over horizon to avoid compounding issues, or simple returns
            group[f'Return_{h}d'] = group['Close'].pct_change(periods=h)
            
            # Volatility scaling
            # Paper uses returns normalized by ex-ante volatility
            group[f'Return_{h}d_scaled'] = group[f'Return_{h}d'] / (group['Vol_60d'] * np.sqrt(h) + 1e-8)
            
        # MACD indicators (8, 24), (16, 28), (32, 96)
        # Paper Eq.4: m̄_{t,s,l} = (v_{t,s} - v_{t,l}) / (sigma_t * sqrt(l))
        # where v_{t,s/l} is the EMA of Close, sigma_t is daily vol
        macd_pairs = [(8, 24), (16, 28), (32, 96)]
        for s, l in macd_pairs:
            group[f'MACD_{s}_{l}'] = compute_macd(group['Close'], s, l)
            # Correct normalisation: EMA difference is in price units, divide by (sigma * Close * sqrt(l))
            # to get a dimensionless signal (volatility-normalised price difference)
            group[f'MACD_{s}_{l}_scaled'] = group[f'MACD_{s}_{l}'] / (group['Close'] * group['Vol_60d'] * np.sqrt(l) + 1e-8)
            
        # Target: next day return
        # Paper uses continuous sizing z_t in (-1, 1). We train to maximize Sharpe.
        # But we need the next day's raw return to calculate the Sharpe loss.
        group['Target_Return'] = group['Return_1d'].shift(-1)
        
        processed_dfs.append(group)
        
    final_df = pd.concat(processed_dfs, ignore_index=True)
    # Drop rows where we don't have enough history for the 252d return or vol
    if keep_unlabeled:
        final_df = final_df.dropna(subset=FEATURE_COLS + ['Vol_60d']).reset_index(drop=True)
    else:
        final_df = final_df.dropna().reset_index(drop=True)
    return final_df

# ==================================================
# portfolio.py
# ==================================================

"""
portfolio.py — Turning per-asset positions into portfolio returns and metrics.

Pure pandas/numpy (no torch), shared by train.py (validation), backtest.py and
live_test.py so all three compute P&L in exactly the same way.

Conventions
-----------
* A "record" is one (Date, Ticker) decision made at the close of Date, with
  z in (-1, 1) from the model and raw_return = the return from that close to
  the next close (Target_Return).
* Per-asset volatility targeting (paper Eq. 9): scaled_z = z * sigma_tgt / sigma_t.
* Transaction cost (paper Eq. 12): cost_bps * |scaled_z_t - scaled_z_{t-1}|,
  computed per ticker in date order.
* Portfolio: equal weight (1/N) across tickers available on each date.
* Portfolio leverage: by default *ex-ante* — uses only portfolio returns up to
  the previous day, so it is tradeable and has no look-ahead.
"""
import numpy as np
import pandas as pd

ANN = 252
ASSET_VOL_TARGET = 0.15      # per-asset annual vol target used in training (paper)
PORTFOLIO_VOL_TARGET = 0.30  # portfolio-level annual vol target for reporting


def scale_positions(z, vol, target_vol=ASSET_VOL_TARGET):
    """Per-asset volatility targeting: z * (daily target vol / daily ex-ante vol)."""
    return z * ((target_vol / np.sqrt(ANN)) / (vol + 1e-8))


def asset_returns(records, cost_bps=3.0, target_vol=ASSET_VOL_TARGET, position_col='z'):
    """
    records: DataFrame with Date, Ticker, raw_return, vol and a position column.
    Returns a copy sorted by (Ticker, Date) with scaled_z, turnover, gross, net.
    The first observed day of each ticker pays the cost of opening its position.
    """
    df = records.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    df['scaled_z'] = scale_positions(df[position_col].values, df['vol'].values, target_vol)
    prev = df.groupby('Ticker')['scaled_z'].shift(1).fillna(0.0)
    df['turnover'] = (df['scaled_z'] - prev).abs()
    df['gross'] = df['scaled_z'] * df['raw_return']
    df['net'] = df['gross'] - df['turnover'] * (cost_bps / 1e4)
    return df


def equal_weight_portfolio(asset_df, col='net'):
    """1/N portfolio: average across the tickers present on each date."""
    return asset_df.groupby('Date')[col].mean().sort_index()


def ex_ante_leverage(portfolio_returns, target_vol=PORTFOLIO_VOL_TARGET,
                     span=60, min_periods=20, max_leverage=None):
    """
    Leverage for day t from an EWMA estimate of portfolio volatility using
    returns up to day t-1 only (shift(1)), so it is known before trading.
    Days without enough history get NaN and are dropped by apply_leverage.
    """
    vol = portfolio_returns.ewm(span=span, min_periods=min_periods).std().shift(1) * np.sqrt(ANN)
    lev = target_vol / (vol + 1e-8)
    if max_leverage is not None:
        lev = lev.clip(upper=max_leverage)
    return lev


def apply_leverage(portfolio_returns, mode='ex_ante', target_vol=PORTFOLIO_VOL_TARGET, **kw):
    """
    mode='ex_ante'  : tradeable, uses only past returns (default).
    mode='constant' : one scale factor from the realised vol of the whole sample.
                      Uses future information; only for presentation (rescaling
                      a finished track record, as the paper does for its plots).
                      Sharpe and Sortino are identical to the unlevered series.
    mode='none'     : no portfolio leverage.
    Returns (levered_returns, leverage) with warm-up days dropped.
    """
    if mode == 'none':
        return portfolio_returns, pd.Series(1.0, index=portfolio_returns.index)
    if mode == 'constant':
        lev_value = target_vol / (portfolio_returns.std() * np.sqrt(ANN) + 1e-8)
        lev = pd.Series(lev_value, index=portfolio_returns.index)
        return portfolio_returns * lev, lev
    if mode == 'ex_ante':
        lev = ex_ante_leverage(portfolio_returns, target_vol, **kw)
        out = (portfolio_returns * lev).dropna()
        return out, lev.loc[out.index]
    raise ValueError(f"unknown leverage mode: {mode}")


def sharpe(returns):
    """Annualised Sharpe ratio of daily returns (rf = 0)."""
    returns = pd.Series(returns).dropna()
    return float(returns.mean() / (returns.std() + 1e-12) * np.sqrt(ANN))


def calculate_metrics(returns, risk_free_rate=0.0):
    """Standard metrics on daily returns (arithmetic annualisation, rf = 0 by default)."""
    returns = pd.Series(returns).dropna()
    ann_return = returns.mean() * ANN
    ann_volatility = returns.std() * np.sqrt(ANN)
    excess = returns - risk_free_rate / ANN
    sharpe_ratio = excess.mean() / (returns.std() + 1e-8) * np.sqrt(ANN)
    # Downside deviation over all days (textbook Sortino denominator)
    downside_dev = np.sqrt(np.mean(np.minimum(returns.values, 0.0) ** 2)) * np.sqrt(ANN)
    sortino = excess.mean() * ANN / (downside_dev + 1e-8)
    cum = (1 + returns).cumprod()
    max_drawdown = ((cum - cum.cummax()) / cum.cummax()).min()
    n_years = len(returns) / ANN
    cagr = cum.iloc[-1] ** (1 / n_years) - 1 if n_years > 0 and cum.iloc[-1] > 0 else np.nan
    return {
        'Annualized Return': ann_return,
        'CAGR': cagr,
        'Annualized Volatility': ann_volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': ann_return / (abs(max_drawdown) + 1e-8),
        'Win Rate': (returns > 0).mean(),
        'Days': len(returns),
    }


def sharpe_standard_error(sharpe_ratio, n_days):
    """Approximate standard error of an annualised Sharpe ratio (Lo, 2002, iid returns)."""
    years = n_days / ANN
    return float(np.sqrt((1 + 0.5 * sharpe_ratio ** 2) / years)) if years > 0 else np.nan


def tsmom_positions(trailing_return):
    """Classic time-series momentum (Moskowitz et al. 2012): long if the 12-month return is positive."""
    return np.sign(trailing_return)

# ==================================================
# dataset.py
# ==================================================

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np



class MomentumDataset(Dataset):
    """
    252-day windows of one ticker's features.

    min_end_date: only keep windows whose LAST day is on or after this date.
        This lets validation/test windows use earlier rows (already-known
        prices) as look-back context, so predictions start on the first day of
        the split instead of 252 trading days later.
    """
    def __init__(self, df, seq_length=252, step=1, min_end_date=None):
        self.seq_length = seq_length
        self.step = step
        self.features = list(FEATURE_COLS)

        self.indices = []
        self.data_X = []
        self.data_y = []
        self.data_vol = []
        self.data_date = []
        self.data_ticker = []

        min_end = pd.Timestamp(min_end_date) if min_end_date is not None else None

        # Group by ticker to prevent sequence overlap between different assets
        for ticker, group in df.groupby('Ticker'):
            group = group.sort_values('Date').reset_index(drop=True)
            n = len(group)
            if n < seq_length:
                continue
            dates = pd.to_datetime(group['Date'])
            first_start = 0
            if min_end is not None:
                # first row on/after min_end, minus the look-back
                first_end = int((dates < min_end).sum())
                if first_end >= n:
                    continue
                first_start = max(0, first_end - (seq_length - 1))

            self.data_X.append(group[self.features].values.astype(np.float32))
            self.data_y.append(group['Target_Return'].values.astype(np.float32))
            self.data_vol.append(group['Vol_60d'].values.astype(np.float32))
            self.data_date.append(dates.dt.strftime('%Y-%m-%d').values)
            self.data_ticker.append(ticker)

            ticker_idx = len(self.data_ticker) - 1
            for i in range(first_start, n - seq_length + 1, self.step):
                self.indices.append((ticker_idx, i))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t_idx, start_idx = self.indices[idx]
        end = start_idx + self.seq_length
        return {
            'X': torch.tensor(self.data_X[t_idx][start_idx:end]),
            'y': torch.tensor(self.data_y[t_idx][start_idx:end]),
            'vol': torch.tensor(self.data_vol[t_idx][start_idx:end]),
            'date': self.data_date[t_idx][end - 1],
            'ticker': self.data_ticker[t_idx]
        }


def split_frames(df, train_end='2018-01-01', val_end='2020-01-01'):
    """
    Time-based split. Validation and test frames include the earlier rows as
    look-back context only; which dates are actually scored is controlled by
    MomentumDataset(min_end_date=...). Features are backward-looking, so using
    already-known earlier prices as context is not leakage.
    """
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    train_df = df[df['Date'] < train_end]
    val_ctx_df = df[df['Date'] < val_end]   # train-period rows used only as context
    test_ctx_df = df                        # val-period rows used only as context
    return train_df, val_ctx_df, test_ctx_df


def get_dataloaders(df, seq_length=252, batch_size=64, train_end='2018-01-01',
                    val_end='2020-01-01', eval_batch_size=256):
    train_df, val_ctx_df, test_ctx_df = split_frames(df, train_end, val_end)

    # Training: non-overlapping windows (paper: start-time step 252 for the TFT)
    train_dataset = MomentumDataset(train_df, seq_length, step=seq_length)
    # Validation / test: one window ending on every trading day of the split
    val_dataset = MomentumDataset(val_ctx_df, seq_length, step=1, min_end_date=train_end)
    test_dataset = MomentumDataset(test_ctx_df, seq_length, step=1, min_end_date=val_end)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=eval_batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# ==================================================
# model.py
# ==================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class GLU(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model)
        self.fc2 = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        return self.fc1(x) * torch.sigmoid(self.fc2(x))

class GRN(nn.Module):
    """Gated Residual Network"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.gate = GLU(d_model)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        res = x
        x = self.fc1(x)
        x = self.elu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.gate(x)
        return self.norm(res + x)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model, dropout=0.1):
        super().__init__()
        self.num_features = num_features
        # We assume input is (batch, seq, num_features). We embed each feature separately.
        # Since our inputs are continuous, we can just use a linear layer for each.
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])
        
        self.flattened_grn = GRN(num_features * d_model, dropout)
        self.softmax_linear = nn.Linear(num_features * d_model, num_features)
        
        self.single_feature_grns = nn.ModuleList([
            GRN(d_model, dropout) for _ in range(num_features)
        ])
        
    def forward(self, x):
        # x shape: (batch, seq, num_features)
        batch_size, seq_len, _ = x.shape
        
        # Embed each feature: output list of (batch, seq, d_model)
        embedded_features = []
        for i in range(self.num_features):
            feat = x[:, :, i].unsqueeze(-1) # (batch, seq, 1)
            embedded = self.feature_embeddings[i](feat)
            embedded_features.append(embedded)
            
        # Concatenate all embedded features
        # (batch, seq, num_features * d_model)
        concat_features = torch.cat(embedded_features, dim=-1)
        
        # Compute variable selection weights
        weight_network = self.flattened_grn(concat_features)
        weights = torch.softmax(self.softmax_linear(weight_network), dim=-1) # (batch, seq, num_features)
        
        # Process each feature through its own GRN and apply weights
        processed_features = []
        for i in range(self.num_features):
            processed = self.single_feature_grns[i](embedded_features[i]) # (batch, seq, d_model)
            weight = weights[:, :, i].unsqueeze(-1) # (batch, seq, 1)
            processed_features.append(processed * weight)
            
        # Sum over features
        # (batch, seq, d_model)
        output = torch.stack(processed_features, dim=-1).sum(dim=-1)
        return output, weights

class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads
        
        self.q_layers = nn.Linear(d_model, d_model)
        self.k_layers = nn.Linear(d_model, d_model)
        # Value weights are shared across heads in TFT (Interpretable MHA)
        self.v_layer = nn.Linear(d_model, self.d_k)
        
        # W_H: output projection from d_k -> d_model (TFT paper requires this)
        self.W_H = nn.Linear(self.d_k, d_model)
        
        self.attention = None
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # Linear projections for Q and K (per-head)
        q = self.q_layers(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_layers(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        # V is SHARED across heads: single projection (batch, seq, d_k)
        v_shared = self.v_layer(v) # (batch, seq, d_k)
        
        # Attention scores (per head)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn = torch.softmax(scores, dim=-1) # (batch, n_heads, seq, seq)
        self.attention = attn # Save for interpretability
        
        # Apply each head's attention to the SHARED V
        # v_shared: (batch, seq, d_k) -> (batch, 1, seq, d_k) for broadcast
        v_expanded = v_shared.unsqueeze(1) # (batch, 1, seq, d_k)
        context_per_head = torch.matmul(attn, v_expanded) # (batch, n_heads, seq, d_k)
        
        # Average across heads (Interpretable MHA: average instead of concatenate)
        context = context_per_head.mean(dim=1) # (batch, seq, d_k)
        
        # W_H output projection
        output = self.W_H(context) # (batch, seq, d_model)
        return output

class MomentumTransformer(nn.Module):
    def __init__(self, num_features=8, d_model=64, n_heads=4, dropout=0.4):
        super().__init__()
        
        self.vsn = VariableSelectionNetwork(num_features, d_model, dropout)
        
        # LSTM encoder for local processing
        self.lstm = nn.LSTM(
            input_size=d_model, 
            hidden_size=d_model, 
            batch_first=True
        )
        
        self.lstm_norm = nn.LayerNorm(d_model)
        self.lstm_gate = GLU(d_model)
        
        # Interpretable Multi-Head Attention
        self.attention = InterpretableMultiHeadAttention(d_model, n_heads)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn_gate = GLU(d_model)
        
        # Feed-forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ELU(),
            nn.Linear(d_model, d_model)
        )
        self.ffn_norm = nn.LayerNorm(d_model)
        self.ffn_gate = GLU(d_model)
        
        # Final output layer predicting position size z_t
        self.output_layer = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ELU(),
            nn.Linear(d_model // 2, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        # x: (batch, seq, num_features)
        
        # 1. Variable Selection Network
        vsn_out, var_weights = self.vsn(x) # (batch, seq, d_model)
        
        # 2. LSTM
        lstm_out, _ = self.lstm(vsn_out) # (batch, seq, d_model)
        # Add & Norm with Gate
        lstm_out = self.lstm_norm(vsn_out + self.lstm_gate(lstm_out))
        
        # 3. Attention (Masked to prevent lookahead)
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(x.device)
        attn_out = self.attention(lstm_out, lstm_out, lstm_out, mask)
        # Add & Norm with Gate
        attn_out = self.attn_norm(lstm_out + self.attn_gate(attn_out))
        
        # 4. FFN
        ffn_out = self.ffn(attn_out)
        out = self.ffn_norm(attn_out + self.ffn_gate(ffn_out))
        
        positions = self.output_layer(out) # (batch, seq, 1)
        z = positions.squeeze(-1) # (batch, seq)
        
        return z, var_weights, self.attention.attention

# ==================================================
# train.py
# ==================================================

import torch
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import os
from tqdm import tqdm

COST = 0.0003   # 3 bps per unit of turnover
SEED = 42


def set_seed(seed=SEED):
    """Make runs repeatable (up to nondeterministic GPU kernels)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sharpe_loss(returns, ann_factor=252):
    """
    Negative annualised Sharpe ratio, the paper's Eq. 11:
    -sqrt(252) * E[R] / std(R) over all asset-time pairs in the batch (Omega).
    Population std (ddof=0) matches the paper's Var[] definition.
    """
    mean_ret = torch.mean(returns)
    std_ret = torch.std(returns, unbiased=False)
    sharpe_ratio = (mean_ret * np.sqrt(ann_factor)) / (std_ret + 1e-8)
    return -sharpe_ratio


def strategy_returns(z, y, vol, cost=COST):
    """
    Per-asset, per-day strategy returns inside each sequence (paper Eq. 9 + 12):
    scaled position = z * sigma_tgt / sigma_t, return = position * next-day return
    minus cost * |change in position|. The first day of a window has no
    previous position inside the window, so no turnover is charged there.
    """
    tgt_vol_daily = ASSET_VOL_TARGET / np.sqrt(252)
    scaled_z = z * (tgt_vol_daily / (vol + 1e-8))
    prev_scaled_z = torch.cat([scaled_z[:, :1], scaled_z[:, :-1]], dim=1)
    turnover = torch.abs(scaled_z - prev_scaled_z)
    return scaled_z * y - cost * turnover


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    epoch_returns = []

    for batch in tqdm(loader, desc="Training"):
        X = batch['X'].to(device)      # (batch, seq, features)
        y = batch['y'].to(device)      # (batch, seq)
        vol = batch['vol'].to(device)  # (batch, seq)

        optimizer.zero_grad()
        z, _, _ = model(X)             # (batch, seq)

        # Flatten into Omega: all asset-time pairs in the batch (Eq. 11)
        omega_returns = strategy_returns(z, y, vol).reshape(-1)
        loss = sharpe_loss(omega_returns)
        loss.backward()

        # Clip gradients (max grad norm 100 is in the paper's TFT search grid)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        optimizer.step()

        total_loss += loss.item()
        epoch_returns.append(omega_returns.detach())

    epoch_returns = torch.cat(epoch_returns)
    epoch_sharpe = -sharpe_loss(epoch_returns).item()
    return total_loss / len(loader), epoch_sharpe


@torch.no_grad()
def predict_last_step(model, loader, device):
    """
    Run every window and keep only its last day: one decision per (Date, Ticker),
    made with the full 252-day look-back, exactly as in the backtest.
    """
    model.eval()
    rows = []
    for batch in loader:
        z, _, _ = model(batch['X'].to(device))
        rows.append(pd.DataFrame({
            'Date': list(batch['date']),
            'Ticker': list(batch['ticker']),
            'z': z[:, -1].cpu().numpy(),
            'raw_return': batch['y'][:, -1].numpy(),
            'vol': batch['vol'][:, -1].numpy(),
        }))
    return pd.concat(rows, ignore_index=True)


def eval_epoch(model, loader, device, cost_bps=COST * 1e4):
    """
    Validation score = Sharpe of the tradeable strategy on the validation period:
    last-step positions, exact per-ticker costs, equal-weight portfolio.
    (Sharpe does not depend on portfolio leverage, so none is applied.)
    """
    preds = predict_last_step(model, loader, device)
    assets = asset_returns(preds, cost_bps=cost_bps)
    return sharpe(equal_weight_portfolio(assets))


def train_model(seed=SEED, features_path="data/features.csv", model_path="models/best_model.pth"):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device} | seed: {seed}")

    df = pd.read_csv(features_path)
    # Paper uses batch size 64 for the TFT (from its grid 32/64/128)
    train_loader, val_loader, _ = get_dataloaders(df, batch_size=64)
    print(f"Train sequences: {len(train_loader.dataset)} | validation decisions: {len(val_loader.dataset)}")

    model = MomentumTransformer(num_features=8, d_model=64, n_heads=4).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_val_sharpe = -float('inf')
    patience = 25
    patience_counter = 0
    epochs = 300

    os.makedirs(os.path.dirname(model_path) or ".", exist_ok=True)

    for epoch in range(epochs):
        train_loss, train_sharpe = train_epoch(model, train_loader, optimizer, device)
        val_sharpe = eval_epoch(model, val_loader, device)

        scheduler.step(val_sharpe)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train Sharpe: {train_sharpe:.4f} | Val Sharpe: {val_sharpe:.4f}")

        if val_sharpe > best_val_sharpe:
            best_val_sharpe = val_sharpe
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            print("  --> Saved new best model!")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    print(f"Training complete. Best validation Sharpe: {best_val_sharpe:.4f}")
    return best_val_sharpe

# ==================================================
# backtest.py
# ==================================================

import json
import os

import numpy as np
import pandas as pd
import torch



def calculate_drawdown(returns):
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    return (cum_returns - running_max) / running_max


def load_model(model_path, device):
    """Load trained weights. Any missing or unexpected key is an error: a silently
    re-initialised layer would make the backtest meaningless."""
    model = MomentumTransformer(num_features=8, d_model=64, n_heads=4)
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"{model_path} not found. Run train.py first.")
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    return model.to(device).eval()


def run_backtest(df_path="data/features.csv", model_path="models/best_model.pth",
                 cost_bps=3, leverage_mode="ex_ante", target_vol=PORTFOLIO_VOL_TARGET,
                 results_dir="results"):
    """
    Out-of-sample backtest on the test split (2020-01-01 onward).

    leverage_mode:
      'ex_ante'  (default) portfolio scaled to `target_vol` using only past returns
                 (60-day EWMA vol, lagged one day). Tradeable, no look-ahead.
      'constant' one scale factor from the whole test period's realised vol.
                 Presentation only: it uses future information. Sharpe/Sortino are
                 unaffected; return, vol and drawdown are "at target vol".
      'none'     raw 1/N portfolio of vol-targeted assets.
    The same leverage method is applied to the TSMOM baseline. The long-only
    benchmark is left unlevered and is compared on Sharpe.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df = pd.read_csv(df_path)
    df['Date'] = pd.to_datetime(df['Date'])

    _, _, test_loader = get_dataloaders(df)
    model = load_model(model_path, device)

    # 1) Model positions: last step of every test window -> one (Date, Ticker) decision
    preds = predict_last_step(model, test_loader, device)
    preds['Date'] = pd.to_datetime(preds['Date'])

    # 2) Per-asset returns with exact per-ticker transaction costs
    strat_assets = asset_returns(preds, cost_bps=cost_bps)
    strat_raw = equal_weight_portfolio(strat_assets)

    # 3) TSMOM baseline on the same dates/tickers: sign of trailing 12-month return
    base = preds[['Date', 'Ticker', 'raw_return', 'vol']].merge(
        df[['Date', 'Ticker', 'Return_252d']], on=['Date', 'Ticker'], how='left')
    base['z'] = tsmom_positions(base['Return_252d'].values)
    tsmom_assets = asset_returns(base, cost_bps=cost_bps)
    tsmom_raw = equal_weight_portfolio(tsmom_assets)

    # 4) Equal-weight long-only benchmark (no costs, unlevered)
    benchmark_returns = preds.groupby('Date')['raw_return'].mean().sort_index()

    # 5) Portfolio leverage
    portfolio_returns, leverage = apply_leverage(strat_raw, mode=leverage_mode, target_vol=target_vol)
    tsmom_returns, _ = apply_leverage(tsmom_raw, mode=leverage_mode, target_vol=target_vol)
    common = portfolio_returns.index.intersection(tsmom_returns.index)
    portfolio_returns, tsmom_returns = portfolio_returns.loc[common], tsmom_returns.loc[common]
    benchmark_returns = benchmark_returns.loc[common]

    metrics = calculate_metrics(portfolio_returns)
    tsmom_metrics = calculate_metrics(tsmom_returns)
    bench_metrics = calculate_metrics(benchmark_returns)

    active = portfolio_returns - benchmark_returns
    ir = active.mean() * 252 / (active.std() * np.sqrt(252) + 1e-8)
    sharpe_se = sharpe_standard_error(metrics['Sharpe Ratio'], metrics['Days'])

    # Cost sensitivity: same positions, different cost assumptions (Sharpe only)
    cost_sens = {c: sharpe(equal_weight_portfolio(asset_returns(preds, cost_bps=c)))
                 for c in [0, 1, 3, 5, 10]}

    start, end = common.min().date(), common.max().date()
    print(f"Out-of-sample period: {start} to {end} ({len(common)} trading days)")
    lev_note = "none" if leverage_mode == "none" else \
        f"{leverage_mode} to {target_vol:.0%} vol (mean leverage {leverage.mean():.2f}x)"
    print(f"Costs: {cost_bps} bps per unit turnover | portfolio leverage: {lev_note}\n")
    table = pd.DataFrame({'Momentum Transformer': metrics, 'TSMOM baseline': tsmom_metrics,
                          'Long-only benchmark': bench_metrics})
    print(table.round(4).to_string())
    print(f"\nInformation ratio vs benchmark: {ir:.4f}")
    print(f"Sharpe standard error (Lo 2002): {sharpe_se:.2f}")
    print("Sharpe by cost assumption: " + ", ".join(f"{c} bps: {s:.2f}" for c, s in cost_sens.items()))

    os.makedirs(results_dir, exist_ok=True)
    portfolio_returns.to_csv(f"{results_dir}/portfolio_returns.csv")
    benchmark_returns.to_csv(f"{results_dir}/benchmark_returns.csv")
    tsmom_returns.to_csv(f"{results_dir}/tsmom_returns.csv")
    summary = {
        'period': [str(start), str(end)], 'trading_days': int(len(common)),
        'cost_bps': cost_bps, 'leverage_mode': leverage_mode, 'target_vol': target_vol,
        'strategy': metrics, 'tsmom': tsmom_metrics, 'benchmark': bench_metrics,
        'information_ratio': ir, 'sharpe_standard_error': sharpe_se,
        'sharpe_by_cost_bps': cost_sens,
    }
    with open(f"{results_dir}/metrics.json", "w") as f:
        json.dump(summary, f, indent=2, default=float)
    write_markdown_summary(table, summary, f"{results_dir}/metrics.md")
    print(f"\nSaved {results_dir}/metrics.md (commit this file to show results on GitHub)")

    return metrics, bench_metrics, portfolio_returns, benchmark_returns


def write_markdown_summary(table, summary, path):
    """A small results table that can be committed to the repo (see .gitignore)."""
    pct_rows = {'Annualized Return', 'CAGR', 'Annualized Volatility', 'Max Drawdown', 'Win Rate'}
    lines = [
        "# Backtest results",
        "",
        f"Out-of-sample: {summary['period'][0]} to {summary['period'][1]} "
        f"({summary['trading_days']} trading days), {summary['cost_bps']} bps costs, "
        f"portfolio leverage `{summary['leverage_mode']}` to {summary['target_vol']:.0%} vol "
        "(benchmark unlevered).",
        "",
        "| Metric | " + " | ".join(table.columns) + " |",
        "|---|" + "---:|" * len(table.columns),
    ]
    for metric, row in table.iterrows():
        cells = []
        for v in row.values:
            if metric == 'Days':
                cells.append(f"{int(v)}")
            elif metric in pct_rows:
                cells.append(f"{v:.2%}")
            else:
                cells.append(f"{v:.2f}")
        lines.append(f"| {metric} | " + " | ".join(cells) + " |")
    lines += [
        "",
        f"Information ratio vs benchmark: {summary['information_ratio']:.2f}. "
        f"Approximate Sharpe standard error: {summary['sharpe_standard_error']:.2f}.",
        "",
        "Sharpe of the unlevered strategy by cost assumption: " +
        ", ".join(f"{c} bps = {s:.2f}" for c, s in summary['sharpe_by_cost_bps'].items()) + ".",
        "",
    ]
    with open(path, "w") as f:
        f.write("\n".join(lines))

# ==================================================
# live_data.py
# ==================================================

"""
live_data.py — Download and feature-engineer fresh data up to today.
Does NOT modify the historical data files. Writes to data/live_raw.csv and
data/live_features.csv. Run after the US market close so the latest bar is final.
"""
import os
from datetime import datetime, timedelta


# A prediction needs 252 rows for the 252-day return feature plus a 252-day
# model window, i.e. about two trading years of history. Starting in 2022
# gives live predictions from early 2024 onward.
LIVE_START = "2022-01-01"


def download_live_data(start=LIVE_START):
    # yfinance's `end` is exclusive, so ask for tomorrow to include today's close
    end = (datetime.today() + timedelta(days=1)).strftime("%Y-%m-%d")
    raw = download_data(TICKERS, start, end, "data/live_raw.csv")
    print(f"Raw live data saved -> data/live_raw.csv  ({len(raw)} rows)")

    # Same feature code as training. keep_unlabeled=True keeps each ticker's
    # latest row (its next-day return is unknown) so today can be traded.
    features = compute_features(raw, keep_unlabeled=True)
    os.makedirs("data", exist_ok=True)
    features.to_csv("data/live_features.csv", index=False)
    print(f"Live features saved -> data/live_features.csv  ({len(features)} rows, "
          f"last date {features['Date'].max().date()})")
    return features

# ==================================================
# live_test.py
# ==================================================

"""
live_test.py -- Run the trained Momentum Transformer on live data (2024 onward).

* Live track record: every day whose next-day return is already known.
* Today's positions: the window ending on the LATEST date (whose return is not
  known yet), so the signal uses today's close, not yesterday's.
Run after the US market close so the latest bar is a final close.
Does NOT modify any training files.
"""
import os
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


SEQ_LENGTH = 252
COST_BPS = 3
RECENT_DAYS = 90
DIRECTION_BAND = 0.15   # display only: |z| below this is shown as FLAT


def direction(z):
    if z > DIRECTION_BAND:
        return "LONG"
    if z < -DIRECTION_BAND:
        return "SHORT"
    return "FLAT"


def run_live_test(features_path="data/live_features.csv", model_path="models/best_model.pth"):
    today_str = datetime.today().strftime("%Y-%m-%d")
    print("=" * 64)
    print(f"  Momentum Transformer - Live Inference [{today_str}]")
    print("=" * 64)

    if not os.path.exists(features_path):
        print("ERROR: Run 'python live_data.py' first.")
        return

    # CPU avoids GPU memory clashes with a running dashboard
    device = torch.device('cpu')
    model = load_model(model_path, device)

    df = pd.read_csv(features_path)
    df['Date'] = pd.to_datetime(df['Date'])
    labelled = df['Target_Return'].notna()
    # The latest row per ticker has no next-day return yet. Fill it with 0 only so
    # the dataset can hold it; that value is never used for the position.
    df_all = df.assign(Target_Return=df['Target_Return'].fillna(0.0))

    dataset = MomentumDataset(df_all, seq_length=SEQ_LENGTH, step=1)
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    print(f"  Running inference on {len(dataset)} windows...")
    preds = predict_last_step(model, loader, device)
    preds['Date'] = pd.to_datetime(preds['Date'])

    unlabelled = df.loc[~labelled, ['Date', 'Ticker']].assign(_latest=True)
    preds = preds.merge(unlabelled, on=['Date', 'Ticker'], how='left')
    is_latest = preds.pop('_latest').fillna(False).astype(bool)
    hist, latest = preds[~is_latest], preds[is_latest]

    # ---- Live track record (decisions whose outcome is known) ----------------
    assets = asset_returns(hist, cost_bps=COST_BPS)
    port_raw = equal_weight_portfolio(assets)
    port, lev = apply_leverage(port_raw, mode="ex_ante", target_vol=PORTFOLIO_VOL_TARGET)
    bench = hist.groupby('Date')['raw_return'].mean().loc[port.index]

    # ---- Today's positions ---------------------------------------------------
    # Leverage for the next session uses every portfolio return realised so far
    vol_now = port_raw.ewm(span=60, min_periods=20).std().iloc[-1] * np.sqrt(ANN)
    lev_today = PORTFOLIO_VOL_TARGET / (vol_now + 1e-8)
    today_pos = latest[['Date', 'Ticker', 'z']].copy()
    today_pos['scaled_z'] = scale_positions(latest['z'].values, latest['vol'].values)
    # Actual portfolio weight: vol-targeted position x portfolio leverage / N
    today_pos['portfolio_weight'] = today_pos['scaled_z'] * lev_today / max(len(today_pos), 1)
    today_pos['Direction'] = today_pos['z'].apply(direction)
    today_pos = today_pos.sort_values('z', ascending=False)

    signal_date = today_pos['Date'].max().date() if len(today_pos) else None
    print("\n" + "-" * 64)
    print(f"  POSITIONS FOR THE NEXT SESSION (signal from the {signal_date} close)")
    print(f"  Portfolio leverage {lev_today:.2f}x to target {PORTFOLIO_VOL_TARGET:.0%} vol")
    print("-" * 64)
    print(f"  {'Ticker':<8} {'Signal z':>9} {'Direction':>10} {'Weight':>9}")
    for _, row in today_pos.iterrows():
        print(f"  {row['Ticker']:<8} {row['z']:>9.4f} {row['Direction']:>10} {row['portfolio_weight']:>8.2%}")

    os.makedirs("results", exist_ok=True)
    today_pos.to_csv("results/live_positions.csv", index=False)
    port.to_csv("results/live_portfolio_returns.csv")
    bench.to_csv("results/live_benchmark_returns.csv")

    # ---- Recent performance (short windows are anecdotal) --------------------
    cutoff = port.index.max() - pd.Timedelta(days=RECENT_DAYS)
    recent_s, recent_b = port[port.index > cutoff], bench[bench.index > cutoff]
    if len(recent_s) > 10:
        sm, bm = calculate_metrics(recent_s), calculate_metrics(recent_b)
        print("\n" + "-" * 64)
        print(f"  LIVE {RECENT_DAYS}-DAY PERFORMANCE ({cutoff.date()} to {port.index.max().date()})")
        print(f"  Strategy levered ex-ante to {PORTFOLIO_VOL_TARGET:.0%} vol; benchmark unlevered")
        print("-" * 64)
        print(f"  {'Metric':<24} {'Strategy':>10} {'Benchmark':>10}")
        for k in ['Annualized Return', 'Annualized Volatility', 'Sharpe Ratio',
                  'Sortino Ratio', 'Max Drawdown', 'Win Rate']:
            sv, bv = sm[k], bm[k]
            if k in ['Annualized Return', 'Annualized Volatility', 'Max Drawdown', 'Win Rate']:
                print(f"  {k:<24} {sv*100:>9.2f}% {bv*100:>9.2f}%")
            else:
                print(f"  {k:<24} {sv:>10.4f} {bv:>10.4f}")

    full = calculate_metrics(port)
    print(f"\n  Full live record since {port.index.min().date()}: Sharpe {full['Sharpe Ratio']:.2f} "
          f"over {full['Days']} days")
    print("\n  Saved -> results/live_positions.csv, live_portfolio_returns.csv, live_benchmark_returns.csv")
    print("=" * 64)



## 4. Download Live Data and Run Inference
Run after the US market close so the latest bar is final.

In [ ]:
live_features = download_live_data()
run_live_test()


## 5. Positions and Live Equity Curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

display(pd.read_csv('results/live_positions.csv'))

port = pd.read_csv('results/live_portfolio_returns.csv', index_col=0, parse_dates=True).iloc[:, 0]
bench = pd.read_csv('results/live_benchmark_returns.csv', index_col=0, parse_dates=True).iloc[:, 0]
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot((1 + port).cumprod(), label='Momentum Transformer (ex-ante 30% vol)', linewidth=2)
ax.plot((1 + bench).cumprod(), label='Equal-weight benchmark (unlevered)', linewidth=2, linestyle='--')
ax.set_title('Live performance since 2024')
ax.set_ylabel('Growth of 1')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/live_equity_curve.png', dpi=150); plt.show()
